# SmartSched: the preference reader, before and after fine-tuning

The model reads a student's survey, notes and messy activity log and writes a preference JSON; the constraint solver turns that JSON into the 3 calendars. This notebook measures how well the model reads, before and after LoRA fine-tuning, on 64 test personas it never trains on (24 of them hand-picked hard cases).

1. On your laptop, from the repo's `ml/` folder:
   `zip -r prefs.zip eval csp prefs -x '*/outputs/*' '*__pycache__*' 'csp/train/*' 'prefs/data/*'`
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** and pick `prefs.zip` when asked.

Roughly: data 10 s, base model runs ~15 min, fine-tuning ~45 min, tuned run ~7 min, end-to-end scoring ~5 min. Keep the tab open.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
# Pick prefs.zip
from google.colab import files
files.upload()
!unzip -oq prefs.zip && ls prefs

In [ ]:
!pip install -q "transformers>=4.56" peft
# Colab ships an old torchao that makes peft refuse to load. We don't use it.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Same seeds every time, so this is the same data as on the laptop.
!python prefs/make_data.py

## 1. Before fine-tuning: the base model, with the rules only and with 2 worked examples

In [ ]:
!python prefs/llm.py --mode zero --model {MODEL} --out preds/base_zero.jsonl

In [ ]:
!python prefs/llm.py --mode few --model {MODEL} --batch 4 --out preds/base_few.jsonl

In [ ]:
!python prefs/evaluate.py preds/base_zero.jsonl preds/base_few.jsonl --out results/before.json

## 2. Fine-tune (LoRA, loss on the reply only)

In [ ]:
import os
!python csp/finetune.py --model {MODEL} --train prefs/data/train.jsonl --val prefs/data/val.jsonl --out adapter_prefs
# Stop Run all here if training failed, instead of running a tuned model that doesn't exist.
assert os.path.exists('adapter_prefs/adapter_config.json'), 'training failed: see the error above'

## 3. After fine-tuning

In [ ]:
!python prefs/llm.py --mode zero --model {MODEL} --adapter adapter_prefs --out preds/tuned.jsonl

## 4. Compare everything

Understanding is scored against the label (what the data supports). End to end, every method's JSON goes through the real scheduler and its picks are judged by the persona's hidden truth: `survey_rules` is the no-model baseline, `oracle` is the best possible.

In [ ]:
!python prefs/evaluate.py preds/base_zero.jsonl preds/base_few.jsonl preds/tuned.jsonl --end-to-end --reports 12 --out results/all.json

In [ ]:
# One persona, every method side by side. Change the name to see another (see prefs/outputs/compare/).
from IPython.display import HTML
HTML(open("prefs/outputs/compare/gym_conflict_1.html").read())

In [ ]:
# Download the replies, results, reports and the adapter
!zip -qr results.zip preds results prefs/outputs adapter_prefs
files.download("results.zip")